<a href="https://colab.research.google.com/github/Amitosh16/Amazon_ML_Challenge_2025/blob/main/Quantile_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
# Mistral Quantile Regression Trainer — Final Version with Auto-Scaled Initialization,
# Diagnostics, Validation SMAPE/MAPE, and Checkpointing

import os, gc, math, random, logging, time
from logging.handlers import RotatingFileHandler
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from torch.nn.utils.rnn import pad_sequence
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split

# =======================
# Config
# =======================
@dataclass
class Config:
    model_name: str = "mistralai/Mistral-7B-Instruct-v0.2"
    train_csv: str = "/content/drive/MyDrive/student_resource/dataset/final_train.csv"
    save_dir: str = "/content/drive/MyDrive/student_resource/result_2"
    resume_from: Optional[str] = None

    val_ratio: float = 0.2
    max_length: int = 192
    batch_size: int = 2
    accum_steps: int = 8
    epochs: int = 4
    lr: float = 5e-6
    lr_head: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    max_grad_norm: float = 0.5
    early_stop_patience: int = 5

    k_quantiles: int = 200
    alpha_smoothing: float = 1e-2
    eval_quantiles: Optional[List[float]] = None

    lora_r: int = 64
    lora_alpha: int = 128
    lora_dropout: float = 0.08

    eval_batch_size: int = 16
    num_workers: int = 2
    seed: int = 42

    def __post_init__(self):
        if self.eval_quantiles is None:
            self.eval_quantiles = [0.25, 0.5, 0.75]

config = Config()
Path(config.save_dir).mkdir(parents=True, exist_ok=True)

# =======================
# Logging
# =======================
log_file = Path(config.save_dir) / "train.log"
logger = logging.getLogger("mistral_qr")
if not logger.handlers:
    logger.setLevel(logging.INFO)
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    ch = logging.StreamHandler()
    ch.setFormatter(fmt)
    fh = RotatingFileHandler(log_file, maxBytes=10_000_000, backupCount=5)
    fh.setFormatter(fmt)
    logger.addHandler(ch)
    logger.addHandler(fh)

METRICS_CSV = Path(config.save_dir) / "metrics.csv"
BEST_CKPT = Path(config.save_dir) / "best.pt"
LAST_CKPT = Path(config.save_dir) / "last.pt"

# =======================
# Setup
# =======================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(config.seed)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    logger.warning("⚠️ CUDA not available — training will be extremely slow.")

# =======================
# Dataset
# =======================
class StreamingTextDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["catalog_content"])
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
            padding=False,
        )
        return enc["input_ids"].squeeze(0), enc["attention_mask"].squeeze(0), torch.tensor(row["log_price"], dtype=torch.float32)

def collate_fn_factory(pad_id: int):
    def collate_fn(batch):
        input_ids = [b[0] for b in batch]
        attn = [b[1] for b in batch]
        y = torch.stack([b[2] for b in batch])
        input_ids_padded = pad_sequence(input_ids, batch_first=True, padding_value=pad_id)
        attn_padded = pad_sequence(attn, batch_first=True, padding_value=0)
        return input_ids_padded, attn_padded, y
    return collate_fn

def load_data(csv_path: str, val_ratio: float):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")
    df = pd.read_csv(csv_path)
    price_col = next((c for c in df.columns if "price" in c.lower()), None)
    df[price_col] = pd.to_numeric(df[price_col], errors="coerce")
    df = df.dropna(subset=["catalog_content", price_col])
    df = df[df[price_col] > 0]
    df["log_price"] = np.log1p(df[price_col])
    df = df[np.isfinite(df["log_price"])]
    logger.info(f"Log price range: [{df['log_price'].min():.2f}, {df['log_price'].max():.2f}]")
    log_price_mean = df["log_price"].mean()
    logger.info(f"Log price mean={log_price_mean:.2f}")
    train_df, val_df = train_test_split(df[["catalog_content", "log_price"]], test_size=val_ratio, random_state=config.seed)
    return train_df, val_df, log_price_mean

# =======================
# Quantile Head (Auto-Scaled)
# =======================
class QuantileHead(nn.Module):
    def __init__(self, hidden_size: int, k: int = 200, target_mean: float = 4.0):
        super().__init__()
        self.k = k
        self.target_mean = target_mean

        self.net = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, k)
        )

        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.02)
                if m.bias is not None:
                    if m == self.net[-1]:
                        bias_init = torch.zeros(k)
                        bias_init[0] = 0.0
                        # auto-scale spread ~3× target_mean, clamped to [4,10]
                        spread_target = min(max(target_mean * 3.0, 4.0), 10.0)
                        delta_target = spread_target / (k - 1)
                        bias_init[1:] = math.log(delta_target + 1e-6)
                        m.bias.data = bias_init
                    else:
                        nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        raw = self.net(x)
        raw = torch.clamp(raw, -10, 10)
        first_q = raw[:, 0:1]
        deltas = torch.nn.functional.softplus(raw[:, 1:])
        deltas = torch.clamp(deltas, max=5.0)
        z = torch.cat([first_q, deltas], dim=1)
        return torch.cumsum(z, dim=1)

# =======================
# Smoothed Pinball Loss
# =======================
class SmoothedPinballLoss(nn.Module):
    def __init__(self, taus: List[float], alpha: float = 1e-2):
        super().__init__()
        self.register_buffer("taus", torch.tensor(taus, dtype=torch.float32))
        self.alpha = float(alpha)

    def forward(self, pred, target):
        taus = self.taus.to(pred.dtype)
        target = target.unsqueeze(1)
        under = torch.clamp((target - pred) / self.alpha, -50, 50)
        over = torch.clamp((pred - target) / self.alpha, -50, 50)
        smooth_under = self.alpha * torch.log1p(torch.exp(under))
        smooth_over = self.alpha * torch.log1p(torch.exp(over))
        return (taus * smooth_under + (1 - taus) * smooth_over).mean()

# =======================
# Quantile Model
# =======================
class QuantileModel(nn.Module):
    def __init__(self, base_model, head):
        super().__init__()
        self.base = base_model
        self.head = head

    def forward(self, ids, mask):
        out = self.base(ids, attention_mask=mask, output_hidden_states=True)
        last_hidden = out.hidden_states[-1]
        seq_lens = mask.sum(dim=1) - 1
        pooled = last_hidden[torch.arange(ids.size(0), device=ids.device), seq_lens]
        return self.head(pooled)

# =======================
# Trainer
# =======================
class Trainer:
    def __init__(self, model, train_loader, val_loader, loss_fn, taus):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.taus = taus
        self.best_smape = float("inf")
        self.patience = 0
        self.epoch = 0

        head_params = [p for n, p in model.named_parameters() if "head" in n]
        base_params = [p for n, p in model.named_parameters() if "head" not in n]
        self.opt = torch.optim.AdamW([
            {"params": base_params, "lr": config.lr},
            {"params": head_params, "lr": config.lr_head}
        ], weight_decay=config.weight_decay)

        total_steps = math.ceil(len(train_loader) / config.accum_steps) * config.epochs
        warmup = int(total_steps * config.warmup_ratio)
        self.scheduler = get_linear_schedule_with_warmup(self.opt, warmup, total_steps)

        self.median_idx = int(np.argmin(np.abs(taus - 0.5)))

    @torch.no_grad()
    def evaluate(self):
        self.model.eval()
        total_smape = total_mape = total_n = 0
        for ids, mask, y in self.val_loader:
            ids, mask, y = ids.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
            with autocast(device_type="cuda", dtype=torch.bfloat16):
                q = self.model(ids, mask)
            preds = torch.expm1(q[:, self.median_idx])
            targets = torch.expm1(y)
            denom = (preds.abs() + targets.abs()) / 2 + 1e-8
            total_smape += (preds - targets).abs().div(denom).sum().item()
            total_mape += (preds - targets).abs().div(targets.abs() + 1e-8).sum().item()
            total_n += len(y)
        if total_n == 0: return float("inf"), float("inf")
        return (total_smape / total_n) * 100, (total_mape / total_n) * 100

    def train(self):
        for epoch in range(config.epochs):
            self.model.train()
            self.opt.zero_grad(set_to_none=True)
            running_loss, steps = 0.0, 0
            pbar = tqdm(self.train_loader, desc=f"Epoch {epoch+1}/{config.epochs}")
            for i, (ids, mask, y) in enumerate(pbar):
                ids, mask, y = ids.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
                with autocast(device_type="cuda", dtype=torch.bfloat16):
                    q = self.model(ids, mask)
                    loss = self.loss_fn(q, y) / config.accum_steps
                loss.backward()
                running_loss += loss.item()
                steps += 1

                if (i + 1) % config.accum_steps == 0 or (i + 1) == len(self.train_loader):
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), config.max_grad_norm)
                    self.opt.step(); self.scheduler.step(); self.opt.zero_grad(set_to_none=True)

                if i % 1000 == 0:
                    spread = (q[:, -1] - q[:, 0]).mean().item()
                    mae = (y - q[:, self.median_idx]).abs().mean().item()
                    logger.info(f"[Epoch {epoch+1}|Step {i}] Loss={running_loss/max(1,steps):.4f} Spread={spread:.2f} MAE={mae:.2f}")

                pbar.set_postfix(loss=f"{running_loss/max(1,steps):.4f}")

            avg_loss = running_loss / steps
            val_smape, val_mape = self.evaluate()
            logger.info(f"Epoch {epoch+1} done | TrainLoss={avg_loss:.4f} | Val SMAPE={val_smape:.2f}% | Val MAPE={val_mape:.2f}%")

            # Early stopping and checkpoints
            if val_smape < self.best_smape:
                self.best_smape = val_smape
                self.patience = 0
                torch.save(self.model.state_dict(), BEST_CKPT)
                logger.info(f"✅ Saved new best model (SMAPE={val_smape:.2f})")
            else:
                self.patience += 1
                torch.save(self.model.state_dict(), LAST_CKPT)

            if self.patience >= config.early_stop_patience:
                logger.info(f"⏹️ Early stopping at epoch {epoch+1}")
                break
            gc.collect(); torch.cuda.empty_cache()

# =======================
# Main
# =======================
def main():
    train_df, val_df, log_price_mean = load_data(config.train_csv, config.val_ratio)
    tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": tokenizer.eos_token})
    pad_id = tokenizer.pad_token_id or 0

    train_ds = StreamingTextDataset(train_df, tokenizer, config.max_length)
    val_ds = StreamingTextDataset(val_df, tokenizer, config.max_length)
    train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True,
                              num_workers=config.num_workers, collate_fn=collate_fn_factory(pad_id))
    val_loader = DataLoader(val_ds, batch_size=config.eval_batch_size, shuffle=False,
                            num_workers=1, collate_fn=collate_fn_factory(pad_id))

    base = AutoModelForCausalLM.from_pretrained(config.model_name, torch_dtype=torch.bfloat16)
    base.gradient_checkpointing_enable()
    if hasattr(base.config, "use_cache"): base.config.use_cache = False
    lora_cfg = LoraConfig(
        r=config.lora_r, lora_alpha=config.lora_alpha, lora_dropout=config.lora_dropout,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        bias="none", task_type="CAUSAL_LM"
    )
    base = get_peft_model(base, lora_cfg).to(DEVICE)
    hidden_size = getattr(base.config, "hidden_size", getattr(base.config, "d_model", None))
    head = QuantileHead(hidden_size, config.k_quantiles, target_mean=log_price_mean).to(DEVICE, dtype=torch.bfloat16)
    model = QuantileModel(base, head)

    logger.info("Testing head initialization...")
    with torch.no_grad():
        dummy_ids = torch.randint(0, 1000, (2, 50)).to(DEVICE)
        dummy_mask = torch.ones_like(dummy_ids)
        q = model(dummy_ids, dummy_mask)
        logger.info(f"Init q range [{q.min():.2f}, {q.max():.2f}] spread={(q[:, -1]-q[:,0]).mean():.2f}")

    taus = np.linspace(0.05, 0.95, config.k_quantiles)
    loss_fn = SmoothedPinballLoss(taus.tolist(), alpha=config.alpha_smoothing).to(DEVICE)
    trainer = Trainer(model, train_loader, val_loader, loss_fn, taus)
    trainer.train()

if __name__ == "__main__":
    main()

2025-10-12 23:08:21,115 | INFO | Log price range: [0.12, 7.94]
INFO:mistral_qr:Log price range: [0.12, 7.94]
2025-10-12 23:08:21,118 | INFO | Log price mean=2.74
INFO:mistral_qr:Log price mean=2.74
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

2025-10-12 23:08:29,927 | INFO | Testing head initialization...
INFO:mistral_qr:Testing head initialization...
2025-10-12 23:08:30,437 | INFO | Init q range [-0.01, 8.12] spread=8.12
INFO:mistral_qr:Init q range [-0.01, 8.12] spread=8.12
Epoch 1/4:   0%|          | 0/29999 [00:00<?, ?it/s]2025-10-12 23:08:31,231 | INFO | [Epoch 1|Step 0] Loss=0.0207 Spread=8.03 MAE=2.48
INFO:mistral_qr:[Epoch 1|Step 0] Loss=0.0207 Spread=8.03 MAE=2.48
Epoch 1/4:   3%|▎         | 1000/29999 [06:34<3:15:21,  2.47it/s, loss=0.0172]2025-10-12 23:15:05,817 | INFO | [Epoch 1|Step 1000] Loss=0.0172 Spread=8.00 MAE=0.87
INFO:mistral_qr:[Epoch 1|Step 1000] Loss=0.0172 Spread=8.00 MAE=0.87
Epoch 1/4:   7%|▋         | 2000/29999 [13:10<3:06:29,  2.50it/s, loss=0.0169]2025-10-12 23:21:41,450 | INFO | [Epoch 1|Step 2000] Loss=0.0169 Spread=7.79 MAE=1.44
INFO:mistral_qr:[Epoch 1|Step 2000] Loss=0.0169 Spread=7.79 MAE=1.44
Epoch 1/4:  10%|█         | 3000/29999 [19:45<2:59:04,  2.51it/s, loss=0.0167]2025-10-12 23:28:

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType
import torch.nn as nn
from tqdm import tqdm

# ===============================================================
# 1️⃣ Paths and setup (UPDATED)
# ===============================================================
base_model_name = "mistralai/Mistral-7B-Instruct-v0.2"
# The single checkpoint file that contains both LoRA and the head
checkpoint_path = "/content/drive/MyDrive/student_resource/result_2/best.pt"
DATASET_FOLDER = "/content/drive/MyDrive/student_resource/dataset"
OUTPUT_FILE = os.path.join(DATASET_FOLDER, "test_out_from_best_pt.csv")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ===============================================================
# 2️⃣ Define Model Architecture (MUST MATCH TRAINING SCRIPT 2)
# ===============================================================

# This QuantileHead class is copied from your *second* training script
class QuantileHead(nn.Module):
    def __init__(self, hidden_size: int, k: int = 200):
        super().__init__()
        self.k = k
        self.net = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, k)
        )

    def forward(self, x):
        raw = self.net(x)
        first_q = raw[:, 0:1]
        deltas = torch.nn.functional.softplus(raw[:, 1:])
        z = torch.cat([first_q, deltas], dim=1)
        return torch.cumsum(z, dim=1)

# This QuantileModel class wraps the base and head
class QuantileModel(nn.Module):
    def __init__(self, base, head):
        super().__init__()
        self.base, self.head = base, head

    def forward(self, ids, mask):
        out = self.base(ids, attention_mask=mask, output_hidden_states=True)
        last_hidden = out.hidden_states[-1]
        # Use the hidden state of the last token for pooling
        seq_lens = mask.sum(dim=1) - 1
        pooled = last_hidden[torch.arange(ids.size(0), device=ids.device), seq_lens]
        return self.head(pooled)

# ===============================================================
# 3️⃣ Build Model Skeleton and Load Weights (NEW LOGIC)
# ===============================================================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Building model skeleton...")
# Step 1: Load the base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16, # Use bfloat16 to match training
    trust_remote_code=True
)

# Step 2: Apply the same LoRA config used in training
lora_cfg = LoraConfig(
    r=64, lora_alpha=128, lora_dropout=0.08,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", task_type=TaskType.CAUSAL_LM
)
peft_model = get_peft_model(base_model, lora_cfg)

# Step 3: Create the head instance
hidden_size = getattr(peft_model.config, "hidden_size", 4096)
head = QuantileHead(hidden_size, k=200)

# Step 4: Combine into the final model structure
qm = QuantileModel(peft_model, head)

print("Loading consolidated checkpoint from best.pt...")
# Step 5: Load the state dict into the complete model structure
qm.load_state_dict(torch.load(checkpoint_path, map_location=device))
qm.to(device)
qm.eval()
print("✅ Model loaded successfully from single checkpoint.")

# ===============================================================
# 4️⃣ Define predictor() function (UPDATED for 200 quantiles)
# ===============================================================

# Calculate the index for the median (50th percentile)
K_QUANTILES = 200
taus = np.linspace(0.05, 0.95, K_QUANTILES)
median_idx = int(np.argmin(np.abs(taus - 0.5)))
print(f"Using index {median_idx} for the median prediction.")

def predictor(sample_id, catalog_content, image_link=None):
    text = str(catalog_content)
    enc = tokenizer(
        text, return_tensors="pt", truncation=True, padding=True, max_length=192
    ).to(device)

    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        out = qm(enc["input_ids"], enc["attention_mask"])
        # Select the median prediction from the 200 outputs
        pred_log_price = out[0, median_idx].cpu().item()

    # Reverse the log1p transform used in the second training script
    price = np.expm1(pred_log_price)

    return float(price)

# ===============================================================
# 5️⃣ Run on test.csv and Save
# ===============================================================
test_path = os.path.join(DATASET_FOLDER, "test.csv")
test = pd.read_csv(test_path)
print(f"Loaded test set: {len(test)} rows")

prices = []
for _, row in tqdm(test.iterrows(), total=len(test), desc="Predicting"):
    price = predictor(row["sample_id"], row["catalog_content"], row.get("image_link", None))
    prices.append(price)

test["price"] = prices
output_df = test[["sample_id", "price"]]
output_df.to_csv(OUTPUT_FILE, index=False)

print(f"\n✅ Predictions saved to: {OUTPUT_FILE}")
print(output_df.head())

Using device: cuda
Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Building model skeleton...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading consolidated checkpoint from best.pt...
✅ Model loaded successfully from single checkpoint.
Using index 99 for the median prediction.
Loaded test set: 75000 rows


Predicting:   0%|          | 0/75000 [00:00<?, ?it/s]/tmp/ipython-input-4046643771.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
Predicting: 100%|██████████| 75000/75000 [2:03:33<00:00, 10.12it/s]


✅ Predictions saved to: /content/drive/MyDrive/student_resource/dataset/test_out_from_best_pt.csv
   sample_id      price
0     100179  10.120411
1     245611  11.676793
2     146263  17.400491
3      95658   4.407590
4      36806  37.068544
